# Phase 3: quantized Qwen3.5-9B, zero-shot (SemEval 2027 Task 6, Task 2)

Does Qwen3.5-9B in NF4 (about 6 GB, one T4 per worker) beat Qwen3.5-4B in NF4 (about 3 GB)? One stage, `qwen9b_z1`, on the
first `ROWS` dev items per track, with exactly the prompt, pixel budget and token cap of the 4B zero-shot control
(`qwen_z1`, Phase 2). Scoring and the judge run locally. Download `phase3_results.zip` from the Output tab.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, Internet **on**.

## 1. Config

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-experiments.git"
BRANCH = "main"
ROWS = 100             # items per track (the first 100 dev rows = the old runs' dev100). 4 = smoke test.
BUDGET_HOURS = 5.0
WORKERS_PER_GPU = 1    # the 9B takes about 6 GB: two workers per T4 would leave too little room for activations
WORK = "/kaggle/working/results"

## 2. Code and dependencies

In [ ]:
import glob, os, subprocess, sys
CODE = "/kaggle/working/mmcqa-experiments"
if not os.path.isdir(CODE):
    local = glob.glob("/kaggle/input/**/mmcqa_exp/__init__.py", recursive=True)    # a Kaggle Dataset copy also works
    if local:
        subprocess.run(["cp", "-r", os.path.dirname(os.path.dirname(local[0])), CODE], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi -L
!pip install -q -r {CODE}/requirements/qwen.txt
if globals().get("INSTALL_FLA", False):        # optional fast kernels for Qwen3.5's linear-attention layers
    !pip install -q flash-linear-attention
    print(subprocess.run([sys.executable, "-c", "import fla; print('fla', fla.__version__)"], capture_output=True, text=True))
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|bitsandbytes|accelerate|huggingface-hub|tokenizers) "

In [ ]:
try:                                   # optional: a Kaggle secret HF_TOKEN avoids anonymous download limits
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: continuing with anonymous downloads")

## 3. Answer with Qwen3.5-9B (NF4)

In [ ]:
!cd {CODE} && python -m mmcqa_exp.phase3 --workdir {WORK} --rows {ROWS} --budget-hours {BUDGET_HOURS} --workers-per-gpu {WORKERS_PER_GPU}

## 4. What happened

In [ ]:
import json, glob
print(open(f"{WORK}/logs/phase3_completion.json").read()[:3000])
for p in sorted(glob.glob(f"{WORK}/predictions/qwen9b_z1/dev_*.jsonl")):
    print(p); [print("  ", json.loads(l)["question"], "->", json.loads(l)["answer"]) for l in open(p).readlines()[:3]]

## 5. Package

In [ ]:
ZIPNAME = "phase3_results.zip"

import shutil, pathlib
media = pathlib.Path(os.environ.get("MMCQA_MEDIA", "/kaggle/temp/mmcqa_media"))
for tmp in (WORK + "/mmcultureqa", "/kaggle/working/mmcqa_media", "/kaggle/working/mmcqa_envs", str(media), str(media.parent / "mmcqa_envs")):
    shutil.rmtree(tmp, ignore_errors=True)
!cd /kaggle/working && zip -qr {ZIPNAME} results && ls -lh {ZIPNAME}